# Data preparation for the part 2 :

In [11]:
from pathlib import Path

dossier = Path('../datasets2')

for fichier in dossier.iterdir():
    if fichier.is_file(): 
        print(fichier.name)

MCTNet_P2_California_Almonds_Seed999.csv
MCTNet_P2_California_Almonds_Seed42.csv
MCTNet_P2_California_Alfalfa_Seed2024.csv
MCTNet_P2_California_Grapes_Seed777.csv
MCTNet_P2_Arkansas_Cotton_Seed2024.csv
MCTNet_P2_California_Rice_Seed777.csv
MCTNet_Arkansas_Corn_Seed42.csv
MCTNet_P2_California_Pistachios_Seed123.csv
MCTNet_P2_Arkansas_Soybeans_Seed999.csv
MCTNet_P2_Arkansas_Cotton_Seed42.csv
MCTNet_Arkansas_Corn_Seed2024.csv
MCTNet_P2_California_Others_Grass_Seed123.csv
MCTNet_P2_California_Pistachios_Seed777.csv
MCTNet_P2_California_Others_Grass_Seed999.csv
MCTNet_P2_Arkansas_Rice_Seed999.csv
MCTNet_P2_Arkansas_Cotton_Seed999.csv
MCTNetP2_California_Rice_Seed314.csv
MCTNetP2_California_Rice_Seed101.csv
MCTNet_P2_Arkansas_Soybeans_Seed42.csv
MCTNet_P2_California_Others_Grass_Seed42.csv
MCTNetP2_California_Rice_Seed555.csv
MCTNet_P2_California_Pistachios_Seed42.csv
MCTNetP2_California_Rice_Seed2024.csv
MCTNet_P2_California_Rice_Seed123.csv
MCTNet_P2_Arkansas_Rice_Seed777.csv
MCTNet_P2_Ark

### concat all csv files into one and drop duplicates

In [12]:
import pandas as pd
from pathlib import Path

dossier_source = Path('../datasets2')
output_cali = 'california_full_unique2.csv'
output_arka = 'arkansas_full_unique2.csv'

def fusion_brute_dedoublonnee(motif, nom_sortie):
    # 1. Collecte de tous les fichiers
    fichiers = list(dossier_source.glob(f'*{motif}*.csv'))
    
    if not fichiers:
        print(f"⚠️ Aucun fichier trouvé pour {motif}")
        return

    print(f"📂 {motif} : Fusion de {len(fichiers)} fichiers en cours...")
    
    # 2. Lecture et concaténation
    liste_df = []
    for f in fichiers:
        try:
            df = pd.read_csv(f)
            # On ignore system:index car il n'est pas unique entre fichiers
            if 'system:index' in df.columns:
                df = df.drop(columns=['system:index'])
            liste_df.append(df)
        except Exception as e:
            print(f"❌ Erreur sur {f.name} : {e}")
    
    df_total = pd.concat(liste_df, ignore_index=True)
    nb_initial = len(df_total)

    # 3. DÉDOUBLONNAGE GÉOGRAPHIQUE
    # On utilise les bandes spectrales de la première date comme empreinte digitale.
    # Si d0_B2, B4, B8, B11 et B12 sont identiques, c'est le même pixel physique.
    signature_spectrale = ['d0_B2', 'd0_B4', 'd0_B8', 'd0_B11', 'd0_B12']
    
    # On s'assure que ces colonnes existent avant de filtrer
    cols_filtre = [c for c in signature_spectrale if c in df_total.columns]
    
    if cols_filtre:
        df_unique = df_total.drop_duplicates(subset=cols_filtre)
    else:
        # Si d0 n'existe pas, on prend les 5 premières colonnes numériques
        cols_filtre = df_total.select_dtypes(include=['number']).columns[:5]
        df_unique = df_total.drop_duplicates(subset=cols_filtre)

    nb_final = len(df_unique)

    # 4. Statistiques de sortie
    print(f"✅ Terminé pour {motif}:")
    print(f"   - Lignes brutes : {nb_initial}")
    print(f"   - Doublons supprimés : {nb_initial - nb_final}")
    print(f"   - Pixels uniques : {nb_final}")
    
    if 'cropland' in df_unique.columns:
        print("📊 Répartition par classe :")
        print(df_unique['cropland'].value_counts().sort_index())

    
    df_unique.to_csv(nom_sortie, index=False)
    print(f"💾 Sauvegardé sous : {nom_sortie}\n")

# Lancement
fusion_brute_dedoublonnee('California', output_cali)
fusion_brute_dedoublonnee('Arkansas', output_arka)

📂 California : Fusion de 27 fichiers en cours...
✅ Terminé pour California:
   - Lignes brutes : 116819
   - Doublons supprimés : 6570
   - Pixels uniques : 110249
📊 Répartition par classe :
cropland
3      25972
36     14843
69     14898
75     14969
76     19588
176    19979
Name: count, dtype: int64
💾 Sauvegardé sous : california_full_unique2.csv

📂 Arkansas : Fusion de 20 fichiers en cours...
✅ Terminé pour Arkansas:
   - Lignes brutes : 84799
   - Doublons supprimés : 382
   - Pixels uniques : 84417
📊 Répartition par classe :
cropland
1      24896
2      24803
3      14972
5       4799
176    14947
Name: count, dtype: int64
💾 Sauvegardé sous : arkansas_full_unique2.csv



### Re-order the columns:

In [13]:
import pandas as pd

def clean_and_reorder_columns(input_file, output_file):
    df = pd.read_csv(input_file)

    # 1. Définir les bandes spectrales
    bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    
    # 2. Ordonner les séries temporelles (d0 à d35)
    ordered_cols = []
    for i in range(36):
        for b in bands:
            col_name = f'd{i}_{b}'
            if col_name in df.columns:
                ordered_cols.append(col_name)
    
    # 3. Ajouter les COVARIABLES (Statiques)
    # On les place après les séries temporelles
    covariates = ['clim_precip', 'clim_temp', 'soil_texture', 'topo_elevation', 'topo_slope']
    for cov in covariates:
        if cov in df.columns:
            ordered_cols.append(cov)

    # 4. Ajouter le LABEL à la toute fin
    if 'cropland' in df.columns:
        ordered_cols.append('cropland')
    
    # On crée le nouveau DataFrame (cela ignore automatiquement '.geo')
    df_final = df[ordered_cols]
 
    df_final.to_csv(output_file, index=False)
    print(f"✅ Terminé : {output_file}")
    print(f"📊 Total colonnes : {len(df_final.columns)} (S2: 360 | Covars: 5 | Label: 1)")

# Exécution
clean_and_reorder_columns('california_full_unique2.csv', 'california.csv')
clean_and_reorder_columns('arkansas_full_unique2.csv', 'arkansas.csv')

✅ Terminé : california.csv
📊 Total colonnes : 366 (S2: 360 | Covars: 5 | Label: 1)
✅ Terminé : arkansas.csv
📊 Total colonnes : 366 (S2: 360 | Covars: 5 | Label: 1)


### California stats:

In [14]:
import pandas as pd


df = pd.read_csv('california.csv')


target_mapping = {
    69: 'Grapes',
    3:  'Rice',
    36: 'Alfalfa',
    75: 'Almonds',
    76: 'Pistachios'
}


objectifs = {
    'Grapes': 2054, 'Rice': 2037, 'Alfalfa': 974, 
    'Almonds': 783, 'Pistachios': 640, 'Others': 3512
}

counts = df['cropland'].value_counts().to_dict()


stats_cibles = {}
others_total = 0
others_details = {}

for code, count in counts.items():
    if code in target_mapping:
        stats_cibles[target_mapping[code]] = count
    else:
        
        others_total += count
        others_details[code] = count

stats_cibles['Others'] = others_total


print(f"{'Crop':<15} | {'Actuel':<8} | {'Objectif':<8} | {'Manquant':<8}")
print("-" * 50)

for name in ['Grapes', 'Rice', 'Alfalfa', 'Almonds', 'Pistachios', 'Others']:
    actual = stats_cibles.get(name, 0)
    target = objectifs.get(name, 0)
    diff = target - actual
    status = f"{diff}" if diff > 0 else "OK"
    print(f"{name:<15} | {actual:<8} | {target:<8} | {status:<8}")

print("-" * 50)
print(f"TOTAL ACTUEL : {sum(stats_cibles.values())}")


Crop            | Actuel   | Objectif | Manquant
--------------------------------------------------
Grapes          | 14898    | 2054     | OK      
Rice            | 25972    | 2037     | OK      
Alfalfa         | 14843    | 974      | OK      
Almonds         | 14969    | 783      | OK      
Pistachios      | 19588    | 640      | OK      
Others          | 19979    | 3512     | OK      
--------------------------------------------------
TOTAL ACTUEL : 110249


### Arkansas stats:

In [3]:
import pandas as pd


df_ark = pd.read_csv('arkansas.csv')

target_mapping_ark = {
    1: 'Soybeans',
    3: 'Rice',
    5: 'Corn',
    2: 'Cotton'
}


objectifs_ark = {
    'Soybeans': 4677, 
    'Rice': 2423, 
    'Corn': 1522, 
    'Cotton': 762, 
    'Others': 616
}


counts_ark = df_ark['cropland'].value_counts().to_dict()


stats_ark = {}
others_total_ark = 0
others_details_ark = {}

for code, count in counts_ark.items():
    if code in target_mapping_ark:
        stats_ark[target_mapping_ark[code]] = count
    else:
        
        others_total_ark += count
        others_details_ark[code] = count

stats_ark['Others'] = others_total_ark


print(f"{'Crop':<15} | {'Actuel':<8} | {'Objectif':<8} | {'Manquant':<8}")
print("-" * 50)

for name in ['Soybeans', 'Rice', 'Corn', 'Cotton', 'Others']:
    actual = stats_ark.get(name, 0)
    target = objectifs_ark.get(name, 0)
    diff = target - actual
    status = f"{diff}" if diff > 0 else "OK"
    print(f"{name:<15} | {actual:<8} | {target:<8} | {status:<8}")

print("-" * 50)
print(f"TOTAL ACTUEL : {sum(stats_ark.values())}")


Crop            | Actuel   | Objectif | Manquant
--------------------------------------------------
Soybeans        | 24896    | 4677     | OK      
Rice            | 14972    | 2423     | OK      
Corn            | 4799     | 1522     | OK      
Cotton          | 24803    | 762      | OK      
Others          | 14947    | 616      | OK      
--------------------------------------------------
TOTAL ACTUEL : 84417


### picking the exact amout of data as the research paper:

In [4]:
import pandas as pd
import glob

def curate_with_exclusion(files_pattern, mapping_targets, goal_counts, other_class_id, output_name):
    print(f"🔄 Traitement de {output_name}...")
    
    
    all_files = glob.glob(files_pattern)
    df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
    
    
    if '.geo' in df.columns: df = df.drop(columns=['.geo'])

 
    df['target_class'] = df['cropland'].apply(lambda x: mapping_targets.get(x, other_class_id))
    
   
    sampled_dfs = []
    for class_id, count in goal_counts.items():
        sub_df = df[df['target_class'] == class_id]
        if len(sub_df) >= count:
            sampled_dfs.append(sub_df.sample(n=count, random_state=42))
        else:
            print(f"⚠️ Warning: Classe {class_id} incomplète ({len(sub_df)}/{count})")
            sampled_dfs.append(sub_df)

    df_final = pd.concat(sampled_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    df_final.to_csv(output_name, index=False)
    print(f"✅ Terminé : {output_name} ({len(df_final)} lignes)")
    print(df_final['target_class'].value_counts().sort_index())


map_ark = {1: 0, 2: 1, 3: 2, 5: 3} 
goals_ark = {0: 4677, 1: 762, 2: 2423, 3: 1522, 4: 616}


map_cal = {69: 0, 3: 1, 36: 2, 75: 3, 76: 4}
goals_cal = {0: 2054, 1: 2037, 2: 974, 3: 783, 4: 640, 5: 3512}


curate_with_exclusion('arkansas.csv', map_ark, goals_ark, 4, 'arkansas_final2.csv')
curate_with_exclusion('california.csv', map_cal, goals_cal, 5, 'california_final2.csv')

🔄 Traitement de arkansas_final2.csv...


/tmp/ipykernel_9768/775995546.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['target_class'] = df['cropland'].apply(lambda x: mapping_targets.get(x, other_class_id))


✅ Terminé : arkansas_final2.csv (10000 lignes)
target_class
0    4677
1     762
2    2423
3    1522
4     616
Name: count, dtype: int64
🔄 Traitement de california_final2.csv...


/tmp/ipykernel_9768/775995546.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['target_class'] = df['cropland'].apply(lambda x: mapping_targets.get(x, other_class_id))


✅ Terminé : california_final2.csv (10000 lignes)
target_class
0    2054
1    2037
2     974
3     783
4     640
5    3512
Name: count, dtype: int64


### Cloud Ratios :

In [1]:
import pandas as pd
import torch
import numpy as np

def analyze_df_quality(df, name="Dataset"):
    
    BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    ordered_cols = []
    for i in range(36):
        for band in BANDS:
            ordered_cols.append(f'd{i}_{band}')
    

    data_raw = df[ordered_cols].values
    x_spectral = torch.tensor(data_raw, dtype=torch.float32).view(-1, 36, 10)
    
   
    days_with_zeros = (x_spectral == 0).any(dim=-1) 
    
   
    total_invalid_days = days_with_zeros.sum().item()
    total_possible_days = x_spectral.shape[0] * 36
    percentage = (total_invalid_days / total_possible_days) * 100
    
    avg_per_sample = days_with_zeros.sum(dim=1).float().mean().item()

    print(f"--- Analyse pour {name} ---")
    print(f"Shape finale : {x_spectral.shape}")
    print(f"Total jours avec au moins un 0 : {total_invalid_days} sur {total_possible_days}")
    print(f"Pourcentage de 'trous' : {percentage:.2f}%")
    print(f"Moyenne de jours impactés par échantillon : {avg_per_sample:.1f} / 36")
    print("-" * 30)
    
    return x_spectral, days_with_zeros

In [3]:
df_cal_final = pd.read_csv('california_final2.csv')
df_ark_final = pd.read_csv('arkansas_final2.csv')
x_cal, holes_cal = analyze_df_quality(df_cal_final, name="California Final")
x_ark, holes_ark = analyze_df_quality(df_ark_final, name="Arkansas Final")

--- Analyse pour California Final ---
Shape finale : torch.Size([10000, 36, 10])
Total jours avec au moins un 0 : 13581 sur 360000
Pourcentage de 'trous' : 3.77%
Moyenne de jours impactés par échantillon : 1.4 / 36
------------------------------
--- Analyse pour Arkansas Final ---
Shape finale : torch.Size([10000, 36, 10])
Total jours avec au moins un 0 : 49082 sur 360000
Pourcentage de 'trous' : 13.63%
Moyenne de jours impactés par échantillon : 4.9 / 36
------------------------------


### Final Preprocessing and normalisation:

In [ ]:
import pandas as pd
import numpy as np

df_california = pd.read_csv('california_final2.csv')
df_arkansas = pd.read_csv('arkansas_final2.csv')


metadata = ['cropland', '.geo', 'target_class'] 
covariables = ['clim_precip', 'clim_temp', 'soil_texture', 'topo_elevation', 'topo_slope']

band_cols = [col for col in df_california.columns if col not in metadata + covariables]


df_california[band_cols] = df_california[band_cols].replace(0, np.nan)
df_arkansas[band_cols] = df_arkansas[band_cols].replace(0, np.nan)

df_california[band_cols] = df_california[band_cols] / 10000
df_arkansas[band_cols] = df_arkansas[band_cols] / 10000

#Z-NORMALISATION DE TOUTES LES COVARIABLES ---
for col in covariables:
    
    df_california[col] = (df_california[col] - df_california[col].mean()) / (df_california[col].std() + 1e-8)
    df_arkansas[col] = (df_arkansas[col] - df_arkansas[col].mean()) / (df_arkansas[col].std() + 1e-8)


df_california = df_california.fillna(0)
df_arkansas = df_arkansas.fillna(0)

df_california.to_csv('california_preprocessed_v2.csv', index=False)
df_arkansas.to_csv('arkansas_preprocessed_v2.csv', index=False)

print("✅ Prétraitement V2 terminé : Bandes S2 (0-1) et Covariables (Z-score).")